# Build the AHBA region x gene matrix (download + processing)

**Step 1 of the adult-brain ASD localization pipeline.** This notebook downloads
the Allen Human Brain Atlas (AHBA) microarray via `abagen` and builds the
Desikan-Killiany region x gene expression matrix that the downstream enrichment /
spin-test notebooks consume. It is split off from the analysis because it needs the
pinned `abagen==0.1.3` environment (pandas<2, numpy<2, setuptools<81) and is slow on
first run; the enrichment notebooks only read the cached CSV it writes.

**Outputs (written to `data/AHBA/`):**
- `ahba_region_by_gene_5donor.csv` - 83 regions x ~15,500 genes
- `ahba_atlas_info.csv` - DK region metadata (id, label, structure, hemisphere)

> **Kernel:** run this notebook in the **`ahba`** conda env (`Python (ahba)`) - it
> holds the pinned `abagen==0.1.3` stack (pandas<2, numpy<2). The downstream
> enrichment notebooks use the lighter `spin` env instead.

**Runtime:** first run downloads ~4 GB and takes 20-40 min; later runs load the cache.

## 1. Setup and environment

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

import abagen, nibabel as nib
print("python     :", sys.version.split()[0])
for m in (np, pd, abagen, nib):
    print(f"{m.__name__:11s}:", m.__version__)

# abagen 0.1.3 compatibility shim: newer pandas removed the `inplace` kwarg on set_axis
_orig_set_axis = pd.DataFrame.set_axis
def _patched_set_axis(self, labels, *args, **kwargs):
    kwargs.pop("inplace", None)
    return _orig_set_axis(self, labels, *args, **kwargs)
pd.DataFrame.set_axis = _patched_set_axis

# abagen requires a writable, absolute data_dir; keep the raw archives under ../data
DATA_DIR = os.path.join(os.path.abspath(os.path.join(os.getcwd(), "..", "..")),
                        "data/AHBA/abagen_data")
os.makedirs(DATA_DIR, exist_ok=True)
print("data_dir   :", DATA_DIR)

## 2. Download the Allen Human Brain Atlas

`abagen.fetch_microarray` pulls each donor's normalized-microarray archive from the
Allen API. We use **five** donors: 9861, 10021, 12876, 14380, 15697. The sixth donor
archive (15496, id 178238266) 404s at the Allen endpoint at analysis time (confirmed
dead on the official download page and in abagen's own donor table), so it is omitted
rather than silently retried.

*First run downloads ~4 GB; subsequent runs read from `DATA_DIR`.*

### Provenance
Raw microarray data come from the **Allen Human Brain Atlas** (Allen Institute for
Brain Science): <https://human.brain-map.org/static/download>. Each archive holds
`MicroarrayExpression.csv`, `SampleAnnot.csv`, `Probes.csv`, `PACall.csv`,
`Ontology.csv`, released under the Allen Institute Terms of Use.

In [ ]:
DONORS = ["9861", "10021", "12876", "14380", "15697"]  # 15496 archive 404s at the Allen API
files = abagen.fetch_microarray(data_dir=DATA_DIR, donors=DONORS, verbose=1)
print("donors fetched:", list(files.keys()))

## 3. Build the region x gene expression matrix

`abagen.get_expression_data` runs the standardized workflow: differential-stability
probe->gene collapse, intensity filtering, sample-to-region assignment on the
Desikan-Killiany atlas, within-donor normalization, and cross-donor aggregation.
Reproducibility options: `lr_mirror="bidirectional"`, `missing="interpolate"`,
`norm_matched=True` (abagen default). Result: 83 regions (68 cortical + subcortex) x
~15,500 genes.

In [ ]:
# Delete data/AHBA/ahba_region_by_gene_5donor.csv to force a clean rebuild.
CACHE = os.path.join(os.path.dirname(DATA_DIR), "ahba_region_by_gene_5donor.csv")
INFO  = os.path.join(os.path.dirname(DATA_DIR), "ahba_atlas_info.csv")
atlas = abagen.fetch_desikan_killiany(surface=False, data_dir=DATA_DIR)

if os.path.exists(CACHE):
    expr = pd.read_csv(CACHE, index_col=0)
    info = pd.read_csv(atlas["info"])
    print("loaded cached matrix:", expr.shape)
else:
    expr = abagen.get_expression_data(
        atlas["image"], atlas["info"], data_dir=DATA_DIR, donors=DONORS,
        lr_mirror="bidirectional", missing="interpolate",
        norm_matched=True, verbose=0)
    info = pd.read_csv(atlas["info"])
    expr.to_csv(CACHE)
    info.to_csv(INFO, index=False)
    print("built matrix:", expr.shape)

info = info.set_index("id")
n_ctx = (info.loc[expr.index, "structure"] == "cortex").sum()
print(f"regions: {expr.shape[0]} ({n_ctx} cortical), genes: {expr.shape[1]}")
print("saved ->", CACHE, "and", INFO)